Bidirectional Encoder Representations from Transformers, or BERT, is a state-of-the art machine learning model used for natural language processing. In 2018, Jacob Devlin and associates at Google created BERT. In 2018, Devlin and his associates trained the BERT using the English Wikipedia (2,500 million words) and BooksCorpus (800 million words), achieving the highest accuracy for certain NLP tasks. Two general BERT versions that have been pre-trained exist: The large model has a neural network design with 24 layers, 1024 hidden layers, 16 heads, and 340 million parameters, whereas the base model has a neural network architecture with 12 layers, 768 hidden layers, and 12 heads.

#REFERENCES


# References
1. https://towardsdatascience.com/sentiment-analysis-in-10-minutes-with-bert-and-hugging-face-294e8a04b671
2. https://medium.com/@dhartidhami/understanding-bert-word-embeddings-7dc4d2ea54ca

#Dataset
3. https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/

4. https://www.kaggle.com/datasets/charunisa/chatgpt-sentiment-analysis?resource=download

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
import torch
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.utils import shuffle
from torch.nn.utils.rnn import pad_sequence


In [ ]:
# Function to read data from Google Drive
def read_data(file_path):
    return pd.read_csv(file_path)

In [ ]:
# Function to convert label to numeric
def convert_label_to_numeric(value):
    if value == 'good':
        return 2
    elif value == 'neutral':
        return 1
    else:
        return 0

In [ ]:
import torch
from torch.utils.data import TensorDataset
from torch.nn.utils.rnn import pad_sequence  # Add this import

# Function to tokenize and format data for BERT
def tokenize_and_format_data(data, tokenizer, max_length=128):
    input_ids = []
    attention_masks = []

    for sentence in data['tweets']:
        encoded_dict = tokenizer.encode_plus(
            sentence,
            add_special_tokens=True,
            max_length=max_length,
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )
        input_ids.append(encoded_dict['input_ids'].squeeze())
        attention_masks.append(encoded_dict['attention_mask'].squeeze())

    # Pad sequences dynamically to the maximum length in the batch
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_masks = pad_sequence(attention_masks, batch_first=True, padding_value=0)

    labels = torch.tensor(data['labels'].values)

    return TensorDataset(input_ids, attention_masks, labels)


In [ ]:
# Function to train the model
def train_model(train_dataloader, model, optimizer, scheduler, device, epochs=2):
    model.train()

    for epoch in tqdm(range(epochs), desc="Epochs"):
        total_loss = 0
        for batch in tqdm(train_dataloader, desc="Batches", leave=False):
            input_ids, attention_mask, labels = batch
            input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        avg_loss = total_loss / len(train_dataloader)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")

In [ ]:
# Function to evaluate the model
def evaluate_model(dataloader, model, device):
    model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids, attention_mask, labels = batch
            input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)

    return all_labels, all_preds



In [ ]:
# Main function
def main():
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # File path on Google Drive
    file_path = '/content/drive/My Drive/sentiment_analysis_with_neural_networks/chatgpt_tweets.csv'

    # Read data
    tweets = read_data(file_path)

    # Preprocess data
    tweets['labels'] = tweets['labels'].apply(convert_label_to_numeric)

    # Split data into training, validation, and test sets
    train_data, test_data = train_test_split(tweets, test_size=0.2, random_state=42)
    train_data, validation_data = train_test_split(train_data, test_size=0.1, random_state=42)

    # Load BERT model and tokenizer
    model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

    # Tokenize and format data for BERT
    max_length = 128
    train_dataset = tokenize_and_format_data(train_data, tokenizer, max_length=max_length)
    validation_dataset = tokenize_and_format_data(validation_data, tokenizer, max_length=max_length)
    test_dataset = tokenize_and_format_data(test_data, tokenizer, max_length=max_length)

    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Create dataloaders
    train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    validation_dataloader = DataLoader(validation_dataset, batch_size=32)
    test_dataloader = DataLoader(test_dataset, batch_size=32)

    # Move model to device
    model.to(device)

    # Set up optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
    total_steps = len(train_dataloader) * 2
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    # Train the model
    train_model(train_dataloader, model, optimizer, scheduler, device)

    # Evaluate on validation set
    val_labels, val_preds = evaluate_model(validation_dataloader, model, device)

    # Print classification report for validation set
    print("Validation Classification Report:")
    print(classification_report(val_labels, val_preds, target_names=['bad','neutral','good']))

    # Evaluate on test set
    test_labels, test_preds = evaluate_model(test_dataloader, model, device)

    # Print classification report for test set
    print("Test Classification Report:")
    print(classification_report(test_labels, test_preds))

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epochs:   0%|          | 0/2 [00:00<?, ?it/s]

...

Epoch 2/2, Loss: 0.1445


Evaluating:   0%|          | 0/549 [00:00<?, ?it/s]

Validation Classification Report:
              precision    recall  f1-score   support

         bad       0.98      0.97      0.98      8762
     neutral       0.93      0.90      0.91      4407
        good       0.94      0.97      0.96      4375

    accuracy                           0.96     17544
   macro avg       0.95      0.95      0.95     17544
weighted avg       0.96      0.96      0.96     17544



Evaluating:   0%|          | 0/1371 [00:00<?, ?it/s]

Test Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.97      0.98     21474
           1       0.92      0.90      0.91     11181
           2       0.94      0.97      0.95     11204

    accuracy                           0.95     43859
   macro avg       0.95      0.95      0.95     43859
weighted avg       0.95      0.95      0.95     43859

